# 04 — Model Gabungan (Time-Series + Fitur NLP)

Notebook ini hanya melatih **model gabungan awal** (Tugas 2, butir 2c): fitur historis kurs + fitur NLP (VADER, Loughran-McDonald, TF-IDF SVD) dengan **XGBoost**.
Model tanpa fitur NLP (Naive Persistence / Majority-Class) sudah dibuat di `src/models/naive_baseline.py` dan `src/models/naive_baseline_regression.py`, jadi tidak dilatih ulang di sini.

Formulasi target mengikuti `docs/target_formulation.md`:

| Task | Kolom target | Model | Metrik primer |
|---|---|---|---|
| Klasifikasi **3-class** | `target` (NAIK / STABIL / TURUN, ambang ±0,1% pada `pct_change`) | `XGBClassifier` | Macro-F1 |
| Regresi | `kurs_besok` (kurs JISDOR hari berikutnya) | `XGBRegressor` | RMSE |

Metrik dihitung dengan `src/models/metrics.py` (modul yang sama dengan naive baseline), supaya angkanya bisa dibandingkan langsung.

**Leakage:** notebook ini tidak melakukan `fit` ulang TF-IDF/SVD (sudah di notebook 03, hanya pada train). Split memakai kolom `split` dari `model_dataset.csv`. Val dipakai untuk early stopping, test hanya dievaluasi sekali di akhir.

## 1. Import dan setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight


def find_repo_root():
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError(
        f"Root repository tidak ditemukan dari {here}. "
        "Jalankan notebook dari dalam repository nlp-trio-bertiga."
    )

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))          # supaya `from src...` bisa dipakai

# Modul milik tim: satu sumber definisi target, metrik, dan naive baseline
from src.jisdor.target_formulation import THRESHOLD_PCT, label_direction
from src.models.metrics import (
    LABEL_ORDER, evaluate_classifier, evaluate_regression,
    directional_hit_from_regression, print_classifier, print_regressor,
)
from src.models.naive_baseline import predict_persistence, predict_majority

FEATURE_DIR = ROOT / "data" / "feature_extracted"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", ROOT)
print("Label kelas  :", LABEL_ORDER)
print("Threshold    : ±%.2f%% pada pct_change harian" % THRESHOLD_PCT)

## 2. Muat dataset

`model_dataset.csv` (keluaran notebook 03) berisi satu baris per hari bursa: fitur kurs, fitur NLP, target, dan kolom `split`.
Target klasifikasi adalah kolom `target` (3-class). Jika kolom itu belum ada di file (mis. `model_dataset.csv` dibuat sebelum notebook 01 memakai `build_target`), `target` dihitung ulang dari `kurs_besok` dengan **aturan yang sama** dengan `build_target()` (`label_direction` dan `THRESHOLD_PCT`), lalu diberi peringatan.

In [ ]:
df = pd.read_csv(FEATURE_DIR / "model_dataset.csv")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

if "target" not in df.columns:
    print("PERINGATAN: kolom `target` tidak ada di model_dataset.csv. "
          "Dihitung ulang dari kurs_besok (aturan sama dengan build_target). "
          "Disarankan menjalankan ulang notebook 01 -> 03.")
    pct_besok = (df["kurs_besok"] / df["kurs_jisdor"] - 1) * 100
    df["target"] = pct_besok.apply(lambda x: label_direction(x, THRESHOLD_PCT))

# Target regresi tetap kurs_besok; delta_besok hanya dipakai sebagai bentuk latihan (lihat bagian 5)
df["delta_besok"] = df["kurs_besok"] - df["kurs_jisdor"]

required = ["date", "kurs_jisdor", "kurs_besok", "target", "split",
            "ret_lag1", "ret_lag2", "ret_lag3", "ret_std5",
            "vader_title_mean", "vader_body_mean", "vader_body_min", "vader_neg_share",
            "lm_negative_mean", "lm_positive_mean", "lm_uncertainty_mean",
            "lm_litigious_mean", "lm_constraining_mean", "lm_net_mean", "no_news"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing}")

tfidf_cols = sorted([c for c in df.columns if c.startswith("tfidf_svd_")],
                    key=lambda x: int(x.split("_")[-1]))

print(f"Total baris : {len(df):,} | kolom: {df.shape[1]}")
print(f"Periode     : {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Fitur TF-IDF SVD: {len(tfidf_cols)} komponen")
print(df["split"].value_counts().reindex(["train", "val", "test"]).to_string())

## 3. Pemeriksaan integritas data

Sel ini tidak mengubah dataset. Ia hanya mendeteksi masalah upstream sebelum model dijalankan: label kelas tidak dikenal, NaN, split yang tumpang tindih, hari akhir pekan (JISDOR hanya terbit di hari kerja), dan lompatan kurs harian yang tidak wajar.

In [ ]:
assert set(df["split"].unique()) <= {"train", "val", "test"}, "Label split tidak dikenal"
assert set(df["target"].unique()) <= set(LABEL_ORDER), f"Label target tidak dikenal: {df['target'].unique()}"

# 1) split harus berurutan kronologis dan tidak tumpang tindih
bounds = df.groupby("split")["date"].agg(["min", "max", "size"]).reindex(["train", "val", "test"])
print(bounds.to_string())
assert bounds.loc["train", "max"] < bounds.loc["val", "min"] < bounds.loc["val", "max"] < bounds.loc["test", "min"], \
    "Split tidak kronologis / tumpang tindih"

# 2) NaN pada fitur dan target
feat_probe = required[1:4] + tfidf_cols
n_nan = int(df[feat_probe].isna().sum().sum())
print("\nNaN pada kolom fitur/target:", n_nan)

# 3) distribusi kelas per split (jadi patokan majority-class)
dist = (df.groupby("split")["target"].value_counts().unstack(fill_value=0)
          .reindex(["train", "val", "test"])[LABEL_ORDER])
print("\nDistribusi kelas target:")
print(dist.to_string())

# 4) rentang kurs per split (pohon tidak bisa memprediksi di luar rentang train)
rng = df.groupby("split")["kurs_jisdor"].agg(["min", "max"]).reindex(["train", "val", "test"])
print("\nRentang kurs per split:")
print(rng.to_string())

# 5) sanity check data kurs
n_weekend = int((df["date"].dt.dayofweek >= 5).sum())
max_jump = (df["kurs_besok"] - df["kurs_jisdor"]).abs().max()
print(f"\nBaris jatuh di akhir pekan : {n_weekend}")
print(f"Lompatan kurs harian maks  : {max_jump:,.0f} rupiah")
if n_weekend > 0 or max_jump > 500:
    print("PERINGATAN: ada tanggal akhir pekan atau lompatan kurs harian sangat besar. "
          "Kemungkinan data kurs di model_dataset.csv salah parse tanggal / tidak sinkron dengan "
          "data/processed/jisdor_target.csv. Jalankan ulang notebook 01 -> 03 sebelum mempercayai hasil.")

## 4. Definisi fitur dan target

Fitur model gabungan = fitur kurs historis (lag return dan volatilitas 5 hari) + VADER + Loughran-McDonald + TF-IDF SVD. Semua fitur hari *t* dipakai untuk memprediksi *t+1*, jadi tidak ada informasi masa depan.
XGBoost berbasis pohon, sehingga tidak perlu standardisasi.

In [ ]:
FEAT_TS = ["ret_lag1", "ret_lag2", "ret_lag3", "ret_std5"]

FEAT_LEXICON = [
    "vader_title_mean", "vader_body_mean", "vader_body_min", "vader_neg_share",
    "lm_negative_mean", "lm_positive_mean", "lm_uncertainty_mean",
    "lm_litigious_mean", "lm_constraining_mean", "lm_net_mean", "no_news",
]

FEAT_COMBINED = FEAT_TS + FEAT_LEXICON + tfidf_cols
print(f"Fitur kurs: {len(FEAT_TS)} | lexicon: {len(FEAT_LEXICON)} | TF-IDF SVD: {len(tfidf_cols)} "
      f"| total: {len(FEAT_COMBINED)}")

# Encoding label 3-class: NAIK=0, STABIL=1, TURUN=2 (urutan sama dengan LABEL_ORDER di metrics.py)
LABEL2ID = {lab: i for i, lab in enumerate(LABEL_ORDER)}
ID2LABEL = {i: lab for lab, i in LABEL2ID.items()}

splits = {s: df[df["split"] == s].sort_values("date").copy() for s in ["train", "val", "test"]}
tr, va, te = splits["train"], splits["val"], splits["test"]

X = {s: d[FEAT_COMBINED] for s, d in splits.items()}
y_clf = {s: d["target"].map(LABEL2ID).astype(int) for s, d in splits.items()}   # klasifikasi
y_reg = {s: d["kurs_besok"].astype(float) for s, d in splits.items()}           # target regresi
y_delta = {s: d["delta_besok"].astype(float) for s, d in splits.items()}        # bentuk latihan regresi
kurs_today = {s: d["kurs_jisdor"].astype(float).to_numpy() for s, d in splits.items()}

print({s: X[s].shape for s in X})

## 5. Referensi naive (tanpa model, tanpa NLP)

Hasil naive baseline di `results/` dihitung pada split `data/split/{train,val,test}.csv`, sedangkan notebook ini memakai split dari `model_dataset.csv` (sudah dipotong ke rentang cakupan berita). Supaya perbandingannya adil, dua referensi dihitung ulang **pada baris yang sama dengan model gabungan**, memakai fungsi `src/models/naive_baseline.py`:

- Klasifikasi: **Persistence of Direction** dan **Majority-Class** (dari train).
- Regresi: **Persistence / Random Walk** (`kurs_besok` = `kurs_jisdor`). RMSE-nya menjadi pembilang skill score.

In [ ]:
all_results = []      # dikumpulkan untuk tabel ringkasan dan JSON

def add_result(model, split, task, metrics):
    all_results.append({"model": model, "split": split, "task": task, **{
        k: v for k, v in metrics.items() if k not in ("classification_report", "confusion_matrix")}})

rmse_naive = {}
for s in ["val", "test"]:
    d = splits[s]
    y_true_lab = d["target"].to_numpy()

    # klasifikasi
    r_p = evaluate_classifier(y_true_lab, predict_persistence(d["target"]), include_kappa=True)
    r_m = evaluate_classifier(y_true_lab, predict_majority(tr["target"], len(d)), include_kappa=True)
    add_result("Naive Persistence of Direction", s, "clf", r_p)
    add_result("Naive Majority-Class", s, "clf", r_m)

    # regresi
    r_r = evaluate_regression(y_reg[s], kurs_today[s], include_mape=True)
    rmse_naive[s] = r_r["rmse"]
    r_r["skill_vs_naive"] = 0.0
    add_result("Naive Persistence (Random Walk)", s, "reg", r_r)

    print(f"[{s}] Persistence-Direction Macro-F1={r_p['macro_f1']:.4f} | "
          f"Majority Macro-F1={r_m['macro_f1']:.4f} | Persistence RMSE={r_r['rmse']:.4f}")

## 6. XGBoost Gabungan: klasifikasi 3-class

Target `target` (NAIK/STABIL/TURUN), `objective="multi:softprob"`. Kelas tidak seimbang, dan kalau tidak dikoreksi XGBoost cenderung selalu menebak kelas terbanyak. Karena itu tiap sampel train diberi bobot `balanced` (`compute_sample_weight`), sesuai metrik primer Macro-F1.
Val dipakai untuk early stopping (`mlogloss`); test tidak disentuh saat fitting.

In [ ]:
def make_xgb_classifier():
    return xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(LABEL_ORDER),
        eval_metric="mlogloss",
        n_estimators=500,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=30,
    )

w_train = compute_sample_weight(class_weight="balanced", y=y_clf["train"])

xgb_clf = make_xgb_classifier()
xgb_clf.fit(
    X["train"], y_clf["train"],
    sample_weight=w_train,
    eval_set=[(X["val"], y_clf["val"])],
    verbose=False,
)
print("Best iteration (clf):", xgb_clf.best_iteration)

pred_clf = {s: pd.Series(xgb_clf.predict(X[s])).map(ID2LABEL).to_numpy() for s in ["val", "test"]}

clf_metrics = {}
for s in ["val", "test"]:
    clf_metrics[s] = evaluate_classifier(splits[s]["target"].to_numpy(), pred_clf[s], include_kappa=True)
    print_classifier(clf_metrics[s], "XGBoost Gabungan (TS+NLP)", s.capitalize())
    add_result("XGBoost Gabungan (TS+NLP)", s, "clf", clf_metrics[s])

## 7. XGBoost Gabungan: regresi `kurs_besok`

Target evaluasi tetap **`kurs_besok`** (nilai kurs mentah besok), sesuai `docs/target_formulation.md`.

**Catatan penting:** model pohon tidak bisa memprediksi di luar rentang nilai yang pernah dilihat di train, padahal kurs pada val/test berada di atas rentang train (lihat tabel rentang kurs di bagian 3). Melatih pohon langsung pada level kurs membuat prediksi "tertahan" di batas atas train dan RMSE membengkak (pada versi lama notebook ini RMSE XGBoost sekitar 1.000 sampai 2.000). Solusinya, model dilatih memprediksi **selisih** `delta_besok = kurs_besok - kurs_jisdor`, lalu dikonversi kembali:

`kurs_besok_prediksi = kurs_jisdor + delta_prediksi`

Dengan begitu titik awalnya adalah persistence (random walk), dan XGBoost hanya belajar koreksi darinya. Target yang dievaluasi tetap `kurs_besok`, jadi RMSE/MAE langsung sebanding dengan naive baseline regresi.

In [ ]:
def make_xgb_regressor():
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=500,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=30,
    )

xgb_reg = make_xgb_regressor()
xgb_reg.fit(
    X["train"], y_delta["train"],
    eval_set=[(X["val"], y_delta["val"])],
    verbose=False,
)
print("Best iteration (reg):", xgb_reg.best_iteration)

# delta -> kurs_besok
pred_reg = {s: kurs_today[s] + xgb_reg.predict(X[s]) for s in ["val", "test"]}

reg_metrics = {}
for s in ["val", "test"]:
    m = evaluate_regression(y_reg[s], pred_reg[s], rmse_naive=rmse_naive[s], include_mape=True)
    m["directional_hit"] = directional_hit_from_regression(kurs_today[s], y_reg[s], pred_reg[s])
    reg_metrics[s] = m
    print_regressor(m, "XGBoost Gabungan (TS+NLP)", s.capitalize())
    add_result("XGBoost Gabungan (TS+NLP)", s, "reg", m)

## 8. Ringkasan hasil

Model gabungan dibandingkan dengan referensi naive pada baris yang sama.
Untuk klasifikasi, model dianggap punya nilai tambah kalau Macro-F1 lebih tinggi dari Persistence dan Majority. Untuk regresi, kalau `skill_vs_naive` > 0 (RMSE lebih kecil dari random walk).

In [ ]:
res = pd.DataFrame(all_results)

clf_cols = ["model", "split", "accuracy", "balanced_accuracy", "macro_f1", "kappa"]
reg_cols = ["model", "split", "rmse", "mae", "mape", "skill_vs_naive", "directional_hit"]
clf_table = res[res["task"] == "clf"][clf_cols].round(4)
reg_table = res[res["task"] == "reg"][reg_cols].round(4)

print("=== KLASIFIKASI 3-class (primer: Macro-F1) ===")
display(clf_table.sort_values(["split", "macro_f1"], ascending=[True, False]))
print("\n=== REGRESI kurs_besok (primer: RMSE) ===")
display(reg_table.sort_values(["split", "rmse"]))

## 9. Confusion matrix (3-class) dan classification report, test set

In [ ]:
cm = np.array(clf_metrics["test"]["confusion_matrix"])
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(LABEL_ORDER)
ax.set_yticks(range(3)); ax.set_yticklabels(LABEL_ORDER)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual")
ax.set_title("XGBoost Gabungan (TS+NLP): Confusion Matrix (Test)")
for i in range(3):
    for j in range(3):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "04_gabungan_confusion_matrix.png", dpi=150)
plt.show()

rep = pd.DataFrame(clf_metrics["test"]["classification_report"]).T.round(4)
display(rep)

## 10. Visualisasi regresi, test set

Kiri: kurs aktual vs prediksi (level). Kanan: selisih `kurs_besok - kurs_jisdor` aktual vs prediksi model; titik ideal berada di garis diagonal. Karena kurs harian mendekati random walk, sebaran di kanan biasanya menempel di dekat nol, dan itu wajar.

In [ ]:
d_te = splits["test"]
actual_delta = y_delta["test"].to_numpy()
pred_delta = pred_reg["test"] - kurs_today["test"]

fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
ax[0].plot(d_te["date"], y_reg["test"], label="Aktual kurs_besok", lw=1.6)
ax[0].plot(d_te["date"], kurs_today["test"], label="Naive Persistence", ls="--", alpha=0.8)
ax[0].plot(d_te["date"], pred_reg["test"], label="XGBoost Gabungan", lw=1.2)
ax[0].set_title("Kurs JISDOR: aktual vs prediksi (Test)")
ax[0].set_ylabel("Rupiah per USD"); ax[0].legend()
ax[0].tick_params(axis="x", rotation=30)

lim = float(max(np.abs(actual_delta).max(), np.abs(pred_delta).max()))
ax[1].scatter(actual_delta, pred_delta, s=14, alpha=0.7)
ax[1].plot([-lim, lim], [-lim, lim], "k--", lw=1)
ax[1].axhline(0, color="grey", lw=0.5); ax[1].axvline(0, color="grey", lw=0.5)
ax[1].set_xlabel("Selisih aktual (rupiah)"); ax[1].set_ylabel("Selisih prediksi (rupiah)")
ax[1].set_title("Selisih kurs besok: aktual vs prediksi (Test)")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "04_gabungan_predicted_vs_actual.png", dpi=150)
plt.show()

## 11. Feature importance

Menunjukkan kontribusi relatif fitur menurut model, **bukan** hubungan kausal. Bagian ini yang paling berguna untuk melihat apakah fitur NLP (VADER, LM, TF-IDF SVD) dipakai model dibanding fitur kurs historis.

In [ ]:
def group_of(f):
    if f in FEAT_TS: return "Kurs historis"
    if f.startswith("tfidf_svd_"): return "TF-IDF SVD"
    return "Lexicon (VADER/LM)"

colors = {"Kurs historis": "#1f3b5c", "Lexicon (VADER/LM)": "#e08a1e", "TF-IDF SVD": "#4c9a6a"}

imp = {
    "Klasifikasi 3-class": pd.Series(xgb_clf.feature_importances_, index=FEAT_COMBINED),
    "Regresi (selisih kurs)": pd.Series(xgb_reg.feature_importances_, index=FEAT_COMBINED),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (title, s) in zip(axes, imp.items()):
    top = s.sort_values(ascending=False).head(15).sort_values()
    ax.barh(top.index, top.values, color=[colors[group_of(f)] for f in top.index])
    ax.set_title(f"Top 15 feature importance: {title}")
    ax.set_xlabel("Importance")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
axes[1].legend(handles, colors.keys(), loc="lower right")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "04_gabungan_feature_importance.png", dpi=150)
plt.show()

share = pd.DataFrame({t: s.groupby(group_of).sum() for t, s in imp.items()}).round(4)
print("Porsi importance per kelompok fitur:")
display(share)

## 12. Simpan hasil

In [ ]:
res.round(6).to_csv(RESULTS_DIR / "04_gabungan_comparison.csv", index=False)
res.round(6).to_csv(RESULTS_DIR / "04_model_comparison.csv", index=False)

out = {
    "model": "XGBoost Gabungan (TS+NLP)",
    "features": {"ts": FEAT_TS, "lexicon": FEAT_LEXICON, "tfidf_svd": len(tfidf_cols)},
    "target": {"classification": "target (NAIK/STABIL/TURUN, threshold %.2f%%)" % THRESHOLD_PCT,
               "regression": "kurs_besok (dilatih pada selisih, dikonversi ke level)"},
    "best_iteration": {"clf": int(xgb_clf.best_iteration), "reg": int(xgb_reg.best_iteration)},
    "classification": {s: {k: v for k, v in m.items()} for s, m in clf_metrics.items()},
    "regression": {s: m for s, m in reg_metrics.items()},
    "rmse_naive_same_rows": rmse_naive,
}
with open(RESULTS_DIR / "04_gabungan_results.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
with open(RESULTS_DIR / "04_all_results.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print("Tersimpan di:", RESULTS_DIR)
for n in ["04_gabungan_comparison.csv", "04_gabungan_results.json", "04_gabungan_confusion_matrix.png",
          "04_gabungan_predicted_vs_actual.png", "04_gabungan_feature_importance.png"]:
    print(" -", n)

## 13. Kesimpulan eksperimen

Isi bagian ini setelah notebook dijalankan dengan data yang sudah diverifikasi (lihat bagian 3). Bahas:

- Macro-F1 model gabungan vs Persistence dan Majority-Class (val dan test); kelas mana yang paling sulit (biasanya TURUN atau STABIL, lihat classification report).
- RMSE, MAE, dan `skill_vs_naive` vs random walk. Skill > 0 berarti ada sinyal di atas persistence; skill ≤ 0 berarti model belum mengalahkannya (hasil yang wajar untuk kurs harian).
- Konsistensi val vs test dan porsi feature importance fitur NLP vs fitur kurs.
- Keterbatasan: data hanya sekitar 1.000 hari, banyak hari tanpa berita (`no_news`), dan TF-IDF SVD berdimensi tinggi untuk jumlah sampel sekecil ini.

Perbedaan performa menunjukkan hubungan prediktif pada dataset/periode eksperimen ini, **bukan** bukti kausal bahwa berita menyebabkan perubahan kurs.